# Compare results

In [4]:
import pandas as pd
import numpy as np

In [11]:
base   = r"..\_output\xgb\xgb_no_t_h_monthly\raw"
regime = r"..\_output\xgb\xgb_no_t_h_test_cp_w_monthly\raw"

b = pd.read_csv(f"{base}/spearman_p.csv",   index_col=0, parse_dates=True)["rho"]
r = pd.read_csv(f"{regime}/spearman_p.csv", index_col=0, parse_dates=True)["rho"]
d = (r - b).dropna()
print(f"base {b.mean():+.4f}  regime {r.mean():+.4f}  "
      f"diff {d.mean():+.4f}  t {d.mean()/(d.std(ddof=1)/np.sqrt(len(d))):.2f}")

base +0.0386  regime +0.0338  diff -0.0048  t -1.47


In [13]:
import pandas as pd, numpy as np

base   = r"..\_output\xgb\xgb_no_t_h_monthly\raw"
regime = r"..\_output\xgb\xgb_no_t_h_test_cp_w_monthly\raw"

# --- 1. did the channel actually fire?  weight distance per rebalance date
wa = pd.read_csv(f"{base}/weights.csv",   index_col=0, parse_dates=True).drop(columns="rebalance_flag")
wb = pd.read_csv(f"{regime}/weights.csv", index_col=0, parse_dates=True).drop(columns="rebalance_flag")
wa, wb = wa.align(wb, join="inner", fill_value=0.0)
dist = (wa - wb).abs().sum(axis=1)
dist = dist[dist.index.isin(b.index)]              # rebalance dates only
print("weight L1 distance per rebalance:")
print(dist.describe().round(4).to_string())
print("\nby year:")
print(dist.groupby(dist.index.year).mean().round(4).to_string())

# --- 2. where does the regime arm lose?  IC difference by year
print("\nIC difference (regime - base) by year:")
print(d.groupby(d.index.year).mean().round(3).to_string())

# --- 3. the 6 months after a regime switch vs. the rest
import sys; sys.path.append(r"..\_regimes")
from changepoint.main import crisis_probs
p = crisis_probs(d.index)["p_crisis"]
switched = (p > 0.5).astype(int).diff().abs().fillna(0).astype(bool)
after = switched.rolling(6, min_periods=1).max().astype(bool)
print(f"\nIC diff, 6m after a switch : {d[after].mean():+.4f}  (n={after.sum()})")
print(f"IC diff, elsewhere        : {d[~after].mean():+.4f}  (n={(~after).sum()})")

weight L1 distance per rebalance:
count    335.0000
mean       0.0679
std        0.0454
min        0.0000
25%        0.0284
50%        0.0652
75%        0.1031
max        0.1894

by year:
date
1998    0.0181
1999    0.0055
2000    0.0400
2001    0.0550
2002    0.0755
2003    0.0510
2004    0.0865
2005    0.0925
2006    0.0935
2007    0.0968
2008    0.0587
2009    0.0737
2010    0.0456
2011    0.1209
2012    0.1003
2013    0.0727
2014    0.1099
2015    0.0934
2016    0.0650
2017    0.1062
2018    0.0549
2019    0.0284
2020    0.0267
2021    0.0494
2022    0.0534
2023    0.0449
2024    0.0942
2025    0.0900

IC difference (regime - base) by year:
date
1998   -0.001
1999    0.002
2000   -0.008
2001   -0.003
2002   -0.003
2003    0.003
2004   -0.010
2005    0.018
2006   -0.013
2007   -0.022
2008    0.001
2009   -0.002
2010   -0.031
2011   -0.029
2012    0.017
2013    0.011
2014   -0.004
2015   -0.020
2016    0.002
2017    0.035
2018    0.016
2019   -0.011
2020   -0.004
2021   -0.004
2022  